In [1]:
import kaggle_environments as ke

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 22.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game_arena
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environment

In [3]:
env = ke.make("orbit_wars", debug=True)
env.reset()
print(env.render(mode="ansi"))

Step 0
Planets:
  ID: 0, Owner: -1, Pos: (93.3, 77.6), R: 1.0, Ships: 6, Prod: 1
  ID: 1, Owner: -1, Pos: (22.4, 93.3), R: 1.0, Ships: 6, Prod: 1
  ID: 2, Owner: -1, Pos: (77.6, 6.7), R: 1.0, Ships: 6, Prod: 1
  ID: 3, Owner: -1, Pos: (6.7, 22.4), R: 1.0, Ships: 6, Prod: 1
  ID: 4, Owner: -1, Pos: (74.4, 95.8), R: 2.6, Ships: 53, Prod: 5
  ID: 5, Owner: -1, Pos: (4.2, 74.4), R: 2.6, Ships: 53, Prod: 5
  ID: 6, Owner: -1, Pos: (95.8, 25.6), R: 2.6, Ships: 53, Prod: 5
  ID: 7, Owner: -1, Pos: (25.6, 4.2), R: 2.6, Ships: 53, Prod: 5
  ID: 8, Owner: -1, Pos: (96.6, 68.5), R: 1.0, Ships: 35, Prod: 1
  ID: 9, Owner: -1, Pos: (31.5, 96.6), R: 1.0, Ships: 35, Prod: 1
  ID: 10, Owner: -1, Pos: (68.5, 3.4), R: 1.0, Ships: 35, Prod: 1
  ID: 11, Owner: -1, Pos: (3.4, 31.5), R: 1.0, Ships: 35, Prod: 1
  ID: 12, Owner: 0, Pos: (82.0, 71.2), R: 1.7, Ships: 10, Prod: 2
  ID: 13, Owner: -1, Pos: (28.8, 82.0), R: 1.7, Ships: 8, Prod: 2
  ID: 14, Owner: -1, Pos: (71.2, 18.0), R: 1.7, Ships: 8, Prod: 2
  

In [4]:
env.step([1, 1])

[{'action': [],
  'reward': 0,
  'info': {},
  'observation': {'remainingOverageTime': 60,
   'step': 1,
   'planets': [[0, -1, 93.26902439076449, 77.56943732543337, 1.0, 6, 1],
    [1, -1, 22.43056267456663, 93.26902439076449, 1.0, 6, 1],
    [2, -1, 77.56943732543337, 6.730975609235514, 1.0, 6, 1],
    [3, -1, 6.730975609235514, 22.43056267456663, 1.0, 6, 1],
    [4, -1, 74.39768821851797, 95.75671884553908, 2.6094379124341005, 53, 5],
    [5, -1, 4.243281154460917, 74.39768821851797, 2.6094379124341005, 53, 5],
    [6, -1, 95.75671884553908, 25.60231178148203, 2.6094379124341005, 53, 5],
    [7, -1, 25.60231178148203, 4.243281154460917, 2.6094379124341005, 53, 5],
    [8, -1, 96.60094089373169, 68.50769804538666, 1.0, 35, 1],
    [9, -1, 31.49230195461334, 96.60094089373169, 1.0, 35, 1],
    [10, -1, 68.50769804538666, 3.399059106268311, 1.0, 35, 1],
    [11, -1, 3.399059106268311, 31.49230195461334, 1.0, 35, 1],
    [12, 0, 82.01609135786202, 71.24598419546959, 1.6931471805599454, 

In [ ]:
import math
import copy
import pandas as pd
import numpy as np


# ── Configuration ─────────────────────────────────────────────────────────────
class GameConfig:
    CENTER = 50.0
    SUN_RADIUS = 10.0
    ROTATION_RADIUS_LIMIT = 50.0
    MAX_SPEED = 6.0
    NB_STEPS_SIM = 10
    PLANET_MARGIN = 0.1
    PLANET_MOVEMENT_SLACK = 3.0


# ── Physics helpers ───────────────────────────────────────────────────────────
class PhysicsEngine:
    @staticmethod
    def distance(p1, p2):
        return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)

    @staticmethod
    def point_to_segment_distance(p, v, w):
        """Minimum distance from point p to line segment v-w."""
        l2 = (v[0] - w[0]) ** 2 + (v[1] - w[1]) ** 2
        if l2 == 0.0:
            return PhysicsEngine.distance(p, v)
        t = max(
            0, min(1, ((p[0] - v[0]) * (w[0] - v[0]) + (p[1] - v[1]) * (w[1] - v[1])) / l2)
        )
        projection = (v[0] + t * (w[0] - v[0]), v[1] + t * (w[1] - v[1]))
        return PhysicsEngine.distance(p, projection)

    @staticmethod
    def swept_pair_hit(A, B, P0, P1, r):
        """True iff a fleet moving A->B and a planet moving P0->P1 come within r
        of each other for some t in [0, 1]."""
        d0x, d0y = A[0] - P0[0], A[1] - P0[1]
        dvx = (B[0] - A[0]) - (P1[0] - P0[0])
        dvy = (B[1] - A[1]) - (P1[1] - P0[1])
        a = dvx * dvx + dvy * dvy
        b = 2.0 * (d0x * dvx + d0y * dvy)
        c = d0x * d0x + d0y * d0y - r * r
        if a < 1e-12:
            return c <= 0.0
        disc = b * b - 4.0 * a * c
        if disc < 0.0:
            return False
        sq = math.sqrt(disc)
        t1 = (-b - sq) / (2.0 * a)
        t2 = (-b + sq) / (2.0 * a)
        return t2 >= 0.0 and t1 <= 1.0

    @staticmethod
    def fleet_speed(ships):
        if ships <= 1:
            return 1.0
        ratio = math.log(ships) / math.log(1000.0)
        return 1.0 + (GameConfig.MAX_SPEED - 1.0) * max(0.0, min(1.0, ratio)) ** 1.5


CENTER = GameConfig.CENTER
SUN_RADIUS = GameConfig.SUN_RADIUS
ROTATION_RADIUS_LIMIT = GameConfig.ROTATION_RADIUS_LIMIT

BOARD_SIZE = 100.0
MAX_NB_STEP = 500

In [ ]:
def interpreter(obs, actions, step, num_agents=2):
    obs0 = obs

    expired_comet_pids = []
    for group in obs0.comets:
        idx = group["path_index"]
        for i, pid in enumerate(group["planet_ids"]):
            if idx >= len(group["paths"][i]):
                expired_comet_pids.append(pid)
    if expired_comet_pids:
        expired_set = set(expired_comet_pids)
        obs0.planets = [p for p in obs0.planets if p[0] not in expired_set]
        obs0.initial_planets = [
            p for p in obs0.initial_planets if p[0] not in expired_set
        ]
        obs0.comet_planet_ids = [
            pid for pid in obs0.comet_planet_ids if pid not in expired_set
        ]
        for group in obs0.comets:
            group["planet_ids"] = [
                pid for pid in group["planet_ids"] if pid not in expired_set
            ]
        obs0.comets = [g for g in obs0.comets if g["planet_ids"]]

    def process_moves(player_id, action):
        if not action or not isinstance(action, list):
            return
        for move in action:
            if len(move) != 3:
                continue
            from_id, angle, ships = move
            ships = int(ships)
            from_planet = next((p for p in obs0.planets if p[0] == from_id), None)
            if from_planet and from_planet[1] == player_id:
                if from_planet[5] >= ships and ships > 0:
                    from_planet[5] -= ships
                    start_x = from_planet[2] + math.cos(angle) * (from_planet[4] + 0.1)
                    start_y = from_planet[3] + math.sin(angle) * (from_planet[4] + 0.1)
                    obs0.fleets.append([
                        obs0.next_fleet_id, player_id,
                        start_x, start_y, angle, from_id, ships,
                    ])
                    obs0.next_fleet_id += 1

    for i in range(num_agents):
        process_moves(i, actions[i])

    for planet in obs0.planets:
        if planet[1] != -1:
            planet[5] += planet[6]

    fleets_to_remove = []
    combat_lists = {p[0]: [] for p in obs0.planets}

    angular_velocity = obs0.angular_velocity
    comet_pid_set = set(obs0.comet_planet_ids)
    initial_by_id = {p[0]: p for p in obs0.initial_planets}

    planet_paths = {}
    for planet in obs0.planets:
        if planet[0] in comet_pid_set:
            continue
        p_old = (planet[2], planet[3])
        p_new = p_old
        initial_p = initial_by_id.get(planet[0])
        if initial_p is not None:
            dx_p = initial_p[2] - CENTER
            dy_p = initial_p[3] - CENTER
            r_p = math.sqrt(dx_p ** 2 + dy_p ** 2)
            if r_p + planet[4] < ROTATION_RADIUS_LIMIT:
                initial_angle = math.atan2(dy_p, dx_p)
                current_angle = initial_angle + angular_velocity * step
                p_new = (
                    CENTER + r_p * math.cos(current_angle),
                    CENTER + r_p * math.sin(current_angle),
                )
        planet_paths[planet[0]] = (p_old, p_new)

    for fleet in obs0.fleets:
        angle = fleet[4]
        ships = fleet[6]
        speed = PhysicsEngine.fleet_speed(ships)
        f_old = (fleet[2], fleet[3])
        fleet[2] += math.cos(angle) * speed
        fleet[3] += math.sin(angle) * speed
        f_new = (fleet[2], fleet[3])

        hit_planet = False
        for planet in obs0.planets:
            path = planet_paths.get(planet[0])
            if path is None:
                continue
            p_old, p_new = path
            if PhysicsEngine.swept_pair_hit(f_old, f_new, p_old, p_new, planet[4]):
                combat_lists[planet[0]].append(fleet)
                fleets_to_remove.append(fleet)
                hit_planet = True
                break
        if hit_planet:
            continue
        if not (0 <= fleet[2] <= BOARD_SIZE and 0 <= fleet[3] <= BOARD_SIZE):
            fleets_to_remove.append(fleet)
            continue
        if PhysicsEngine.point_to_segment_distance((CENTER, CENTER), f_old, f_new) < SUN_RADIUS:
            fleets_to_remove.append(fleet)
            continue

    for planet in obs0.planets:
        path = planet_paths.get(planet[0])
        if path is not None:
            planet[2], planet[3] = path[1]

    expired_comet_pids = []
    for group in obs0.comets:
        group["path_index"] += 1
        idx = group["path_index"]
        for i, pid in enumerate(group["planet_ids"]):
            planet = next((p for p in obs0.planets if p[0] == pid), None)
            if planet is None:
                continue
            p_path = group["paths"][i]
            if idx >= len(p_path):
                expired_comet_pids.append(pid)
            else:
                c_old = (planet[2], planet[3])
                planet[2] = p_path[idx][0]
                planet[3] = p_path[idx][1]
                if c_old[0] >= 0:
                    c_new = (planet[2], planet[3])
                    for fleet in obs0.fleets:
                        if fleet not in fleets_to_remove:
                            if PhysicsEngine.point_to_segment_distance((fleet[2], fleet[3]), c_old, c_new) < planet[4]:
                                combat_lists[planet[0]].append(fleet)
                                fleets_to_remove.append(fleet)

    if expired_comet_pids:
        expired_set = set(expired_comet_pids)
        obs0.planets = [p for p in obs0.planets if p[0] not in expired_set]
        obs0.initial_planets = [p for p in obs0.initial_planets if p[0] not in expired_set]
        obs0.comet_planet_ids = [pid for pid in obs0.comet_planet_ids if pid not in expired_set]
        for group in obs0.comets:
            group["planet_ids"] = [pid for pid in group["planet_ids"] if pid not in expired_set]
        obs0.comets = [g for g in obs0.comets if g["planet_ids"]]

    obs0.fleets = [f for f in obs0.fleets if f not in fleets_to_remove]

    for pid, planet_fleets in combat_lists.items():
        planet = next((p for p in obs0.planets if p[0] == pid), None)
        if not planet or not planet_fleets:
            continue
        player_ships = {}
        for fleet in planet_fleets:
            owner = fleet[1]
            player_ships[owner] = player_ships.get(owner, 0) + fleet[6]
        if not player_ships:
            continue
        sorted_players = sorted(player_ships.items(), key=lambda item: item[1], reverse=True)
        top_player, top_ships = sorted_players[0]
        if len(sorted_players) > 1:
            second_ships = sorted_players[1][1]
            survivor_ships = top_ships - second_ships
            if sorted_players[0][1] == sorted_players[1][1]:
                survivor_ships = 0
            survivor_owner = top_player if survivor_ships > 0 else -1
        else:
            survivor_owner = top_player
            survivor_ships = top_ships
        if survivor_ships > 0:
            if planet[1] == survivor_owner:
                planet[5] += survivor_ships
            else:
                planet[5] -= survivor_ships
                if planet[5] < 0:
                    planet[1] = survivor_owner
                    planet[5] = abs(planet[5])

    obs1 = {
        "planets": obs0.planets,
        "initial_planets": obs0.initial_planets,
        "fleets": obs0.fleets,
        "next_fleet_id": obs0.next_fleet_id,
        "comets": obs0.comets,
        "comet_planet_ids": obs0.comet_planet_ids,
    }

    terminated = False
    if step >= MAX_NB_STEP - 2:
        terminated = True
    alive_players = set()
    for p in obs0.planets:
        if p[1] != -1:
            alive_players.add(p[1])
    for f in obs0.fleets:
        alive_players.add(f[1])
    if len(alive_players) <= 1:
        terminated = True

    return obs1

In [ ]:
class StrategyPipeline:

    @staticmethod
    def _01_get_obs_dataframe(obs, step: int, num_agents: int) -> tuple:
        sim = copy.deepcopy(obs)
        no_actions = [[] for _ in range(num_agents)]
        rows = []
        for i in range(GameConfig.NB_STEPS_SIM + 1):
            for p in sim.planets:
                pid, owner, x, y, radius, ships, production = (
                    p[0], p[1], p[2], p[3], p[4], p[5], p[6]
                )
                r = math.hypot(x - GameConfig.CENTER, y - GameConfig.CENTER)
                if pid in sim.comet_planet_ids:
                    nature = "comet"
                elif r + radius < GameConfig.ROTATION_RADIUS_LIMIT:
                    nature = "moving"
                else:
                    nature = "fix"
                rows.append({
                    "step": step + i,
                    "id": pid,
                    "x": x,
                    "y": y,
                    "radius": radius,
                    "ships": ships,
                    "production": production,
                    "owner": owner,
                    "nature": nature,
                })
            interpreter(sim, no_actions, step + i, num_agents)

        df_s = pd.DataFrame(rows).sort_values("step").reset_index(drop=True)

        prev_pos = (
            df_s[["id", "step", "x", "y"]]
            .assign(step=lambda d: d["step"] + 1)
            .rename(columns={"x": "x_prev", "y": "y_prev"})
        )
        planet_disp = (
            df_s[["id", "step", "x", "y"]]
            .merge(prev_pos, on=["id", "step"], how="left")
            .assign(
                planet_disp=lambda d: np.sqrt(
                    (d["x"] - d["x_prev"].fillna(d["x"])) ** 2 +
                    (d["y"] - d["y_prev"].fillna(d["y"])) ** 2
                )
            )
            [["id", "step", "planet_disp"]]
        )
        return df_s, planet_disp

    @staticmethod
    def _02_get_all_opportunities(
        df_s: pd.DataFrame,
        planet_disp: pd.DataFrame,
        player_id: int,
    ) -> pd.DataFrame:
        # Per-step source data for always-mine planets
        always_mine_mask = (
            df_s.groupby("id")["owner"]
            .transform(lambda g: (g == player_id).all())
        )
        always_mine_ids = df_s.loc[always_mine_mask, "id"].unique()

        if len(always_mine_ids) == 0:
            return pd.DataFrame()

        mine_per_step = (
            df_s
            .loc[df_s["id"].isin(always_mine_ids)]
            .sort_values(["id", "step"])
            .assign(
                ships_min=lambda d: d.groupby("id")["ships"]
                                     .transform(lambda s: s[::-1].cummin()[::-1])
            )
            .rename(columns={
                "id": "id_src",
                "step": "step_src",
                "x": "x_src",
                "y": "y_src",
                "radius": "radius_src",
                "ships": "ships_src",
                "production": "production_src",
                "nature": "nature_src",
                "owner": "owner_src",
            })
            .reset_index(drop=True)
        )

        df_tgt = df_s.rename(columns={
            "id": "id_tgt",
            "step": "step_tgt",
            "x": "x_tgt",
            "y": "y_tgt",
            "radius": "radius_tgt",
            "ships": "ships_tgt",
            "production": "production_tgt",
            "nature": "nature_tgt",
            "owner": "owner_tgt",
        })

        # Phase A: planet-level cross join
        coarse = (
            mine_per_step.assign(_key=1)
            .merge(df_tgt.assign(_key=1), on="_key")
            .drop(columns="_key")
            .loc[lambda d: (d["step_tgt"] > d["step_src"]) & (d["id_tgt"] != d["id_src"])]
            .merge(
                planet_disp.rename(columns={"id": "id_tgt", "step": "step_tgt"}),
                on=["id_tgt", "step_tgt"], how="left"
            )
            .reset_index(drop=True)
            .assign(
                dist_tgt_src=lambda d: np.sqrt(
                    (d["x_tgt"] - d["x_src"]) ** 2 + (d["y_tgt"] - d["y_src"]) ** 2
                ),
                step_diff=lambda d: (d["step_tgt"] - d["step_src"]).astype(float),
            )
        )

        # Sun-crossing filter (vectorised)
        _dx = coarse["x_tgt"].values - coarse["x_src"].values
        _dy = coarse["y_tgt"].values - coarse["y_src"].values
        _l2 = _dx ** 2 + _dy ** 2
        _dot = (GameConfig.CENTER - coarse["x_src"].values) * _dx + (GameConfig.CENTER - coarse["y_src"].values) * _dy
        _t_sun = np.clip(_dot / np.where(_l2 == 0, 1.0, _l2), 0.0, 1.0)
        _proj = np.sqrt(
            (GameConfig.CENTER - coarse["x_src"].values - _t_sun * _dx) ** 2 +
            (GameConfig.CENTER - coarse["y_src"].values - _t_sun * _dy) ** 2
        )
        _sun_dist = np.where(
            _l2 == 0,
            np.sqrt((GameConfig.CENTER - coarse["x_src"].values) ** 2 + (GameConfig.CENTER - coarse["y_src"].values) ** 2),
            _proj,
        )
        _crossing_sun = _sun_dist < (GameConfig.SUN_RADIUS + GameConfig.PLANET_MARGIN)

        coarse = (
            coarse
            .assign(_crossing_sun=_crossing_sun)
            .loc[lambda d:
                (d["dist_tgt_src"] <
                 (d["step_diff"] + 1) * GameConfig.MAX_SPEED
                 + d["radius_src"] + GameConfig.PLANET_MARGIN + d["radius_tgt"]
                 + d["planet_disp"].fillna(0.0))
                & ~d["_crossing_sun"]
            ]
            .drop(columns="_crossing_sun")
            .reset_index(drop=True)
        )

        if coarse.empty:
            return pd.DataFrame()

        # Ships_sent expansion
        expanded = (
            coarse
            .assign(
                ships_sent=lambda d: [
                    list(range(1, int(sm) + 1))
                    for sm in d["ships_min"]
                ]
            )
            .explode("ships_sent")
            .assign(ships_sent=lambda d: d["ships_sent"].astype("int64"))
            .reset_index(drop=True)
        )

        # Phase B: fleet-speed filter
        _fs_ratio = np.clip(
            np.log(expanded["ships_sent"].values.astype(float)) / math.log(1000.0),
            0, None,
        )
        _fleet_speed_b = 1.0 + (GameConfig.MAX_SPEED - 1.0) * _fs_ratio ** 1.5
        _dist_min_b = expanded["step_diff"].values * _fleet_speed_b + GameConfig.PLANET_MARGIN + expanded["radius_src"].values
        _dist_prev_b = _dist_min_b - _fleet_speed_b

        prev_pos_tgt = (
            df_s[["id", "step", "x", "y"]]
            .assign(step=lambda d: d["step"] + 1)
            .rename(columns={"id": "id_tgt", "step": "step_tgt", "x": "x_prev_tgt", "y": "y_prev_tgt"})
        )

        expanded = (
            expanded
            .assign(fleet_speed=_fleet_speed_b, dist_min=_dist_min_b, dist_prev=_dist_prev_b)
            .loc[lambda d: d["dist_tgt_src"] < d["dist_min"] + d["fleet_speed"] + d["radius_tgt"] + GameConfig.PLANET_MOVEMENT_SLACK]
            .merge(prev_pos_tgt, on=["id_tgt", "step_tgt"], how="left")
            .reset_index(drop=True)
        )

        if expanded.empty:
            return pd.DataFrame()

        # Swept-pair collision (vectorised)
        _dx2 = expanded["x_tgt"].values - expanded["x_src"].values
        _dy2 = expanded["y_tgt"].values - expanded["y_src"].values
        _dist2 = expanded["dist_tgt_src"].values
        _ux = _dx2 / np.where(_dist2 < 1e-9, 1.0, _dist2)
        _uy = _dy2 / np.where(_dist2 < 1e-9, 1.0, _dist2)

        _xpf = expanded["x_prev_tgt"].fillna(expanded["x_tgt"]).values
        _ypf = expanded["y_prev_tgt"].fillna(expanded["y_tgt"]).values

        _fx0 = expanded["x_src"].values + _ux * expanded["dist_prev"].values
        _fy0 = expanded["y_src"].values + _uy * expanded["dist_prev"].values
        _pvx = expanded["x_tgt"].values - _xpf
        _pvy = expanded["y_tgt"].values - _ypf
        _dvx = _ux * expanded["fleet_speed"].values - _pvx
        _dvy = _uy * expanded["fleet_speed"].values - _pvy
        _d0x = _fx0 - _xpf
        _d0y = _fy0 - _ypf
        _a   = _dvx ** 2 + _dvy ** 2
        _b_  = 2.0 * (_d0x * _dvx + _d0y * _dvy)
        _c_  = _d0x ** 2 + _d0y ** 2 - expanded["radius_tgt"].values ** 2
        _disc = _b_ ** 2 - 4.0 * _a * _c_
        _sq  = np.sqrt(np.clip(_disc, 0, None))
        _t1  = np.where(_a < 1e-12, 0.0, (-_b_ - _sq) / (2.0 * _a))
        _t2  = np.where(_a < 1e-12, 1.0, (-_b_ + _sq) / (2.0 * _a))
        _coll = np.where(_a < 1e-12, _c_ <= 0.0, (_disc >= 0.0) & (_t2 >= 0.0) & (_t1 <= 1.0))

        pa = (
            expanded
            .assign(t1=_t1, t2=_t2, collision=_coll)
            .loc[lambda d: d["collision"]]
            .reset_index(drop=True)
        )

        if pa.empty:
            return pa

        # Angle geometry
        _xpf_pa = pa["x_prev_tgt"].fillna(pa["x_tgt"]).values
        _ypf_pa = pa["y_prev_tgt"].fillna(pa["y_tgt"]).values
        _t1e = np.clip(pa["t1"].values, 0.0, 1.0)
        _t2e = np.clip(pa["t2"].values, 0.0, 1.0)

        pa = (
            pa
            .assign(
                t1_eff=_t1e,
                t2_eff=_t2e,
                p_t1_x=_xpf_pa + _t1e * (pa["x_tgt"].values - _xpf_pa),
                p_t1_y=_ypf_pa + _t1e * (pa["y_tgt"].values - _ypf_pa),
                p_t2_x=_xpf_pa + _t2e * (pa["x_tgt"].values - _xpf_pa),
                p_t2_y=_ypf_pa + _t2e * (pa["y_tgt"].values - _ypf_pa),
            )
            .assign(
                angle_t1=lambda d: np.arctan2(d["p_t1_y"] - d["y_src"], d["p_t1_x"] - d["x_src"]),
                angle_t2=lambda d: np.arctan2(d["p_t2_y"] - d["y_src"], d["p_t2_x"] - d["x_src"]),
                d_s_t1=lambda d: np.sqrt((d["p_t1_x"] - d["x_src"]) ** 2 + (d["p_t1_y"] - d["y_src"]) ** 2),
                d_s_t2=lambda d: np.sqrt((d["p_t2_x"] - d["x_src"]) ** 2 + (d["p_t2_y"] - d["y_src"]) ** 2),
            )
            .assign(
                d_f_t1=lambda d: d["dist_prev"] + d["t1_eff"] * d["fleet_speed"],
                d_f_t2=lambda d: d["dist_prev"] + d["t2_eff"] * d["fleet_speed"],
            )
            .assign(
                angle_radius_t1=lambda d: np.arccos(np.clip(
                    (d["d_s_t1"] ** 2 + d["d_f_t1"] ** 2 - d["radius_tgt"] ** 2)
                    / (2.0 * d["d_s_t1"] * d["d_f_t1"]),
                    -1.0, 1.0,
                )),
                angle_radius_t2=lambda d: np.arccos(np.clip(
                    (d["d_s_t2"] ** 2 + d["d_f_t2"] ** 2 - d["radius_tgt"] ** 2)
                    / (2.0 * d["d_s_t2"] * d["d_f_t2"]),
                    -1.0, 1.0,
                )),
            )
            .assign(
                angle_min=lambda d: np.minimum(
                    d["angle_t1"] - d["angle_radius_t1"],
                    d["angle_t2"] - d["angle_radius_t2"],
                ) % (2 * math.pi),
                angle_max=lambda d: np.maximum(
                    d["angle_t1"] + d["angle_radius_t1"],
                    d["angle_t2"] + d["angle_radius_t2"],
                ) % (2 * math.pi),
                angle=lambda d: np.arctan2(
                    np.sin(d["angle_t1"]) + np.sin(d["angle_t2"]),
                    np.cos(d["angle_t1"]) + np.cos(d["angle_t2"]),
                ),
            )
            .sort_values("step_tgt")
            .reset_index(drop=True)
        )

        return pa

    @staticmethod
    def _03_filter_collision(pa: pd.DataFrame) -> pd.DataFrame:
        if pa.empty:
            return pa

        pa_left = pa[["id_src", "ships_sent", "step_tgt", "id_tgt", "angle", "angle_min", "angle_max"]].copy()
        pa_obs = (
            pa[["id_src", "ships_sent", "step_tgt", "id_tgt", "angle_min", "angle_max"]]
            .rename(columns={
                "step_tgt": "step_tgt_obs",
                "id_tgt": "id_tgt_obs",
                "angle_min": "angle_min_obs",
                "angle_max": "angle_max_obs",
            })
        )

        blocked_joined = (
            pa_left
            .merge(pa_obs, on=["id_src", "ships_sent"])
            .loc[lambda d: (d["step_tgt_obs"] < d["step_tgt"]) & (d["id_tgt_obs"] != d["id_tgt"])]
            .reset_index(drop=True)
        )

        if not blocked_joined.empty:
            _anorm = blocked_joined["angle"].values % (2 * math.pi)
            _wraps = (blocked_joined["angle_min_obs"] > blocked_joined["angle_max_obs"]).values
            _in_cone = np.where(
                _wraps,
                (_anorm >= blocked_joined["angle_min_obs"].values) | (_anorm <= blocked_joined["angle_max_obs"].values),
                (_anorm >= blocked_joined["angle_min_obs"].values) & (_anorm <= blocked_joined["angle_max_obs"].values),
            )
            blocked = (
                blocked_joined[_in_cone]
                [["id_src", "ships_sent", "step_tgt", "id_tgt"]]
                .drop_duplicates()
            )
        else:
            blocked = pd.DataFrame(columns=["id_src", "ships_sent", "step_tgt", "id_tgt"])

        attacks_with_angle = (
            pa
            .merge(blocked.assign(_blocked=True), on=["id_src", "ships_sent", "step_tgt", "id_tgt"], how="left")
            .loc[lambda d: d["_blocked"].isna()]
            .drop(columns="_blocked")
            .assign(final_angle=lambda d: d["angle"])
            .reset_index(drop=True)
        )

        return attacks_with_angle

In [ ]:
obs = env.state[0].observation
step = 0
num_agents = 2
df_s, planet_disp = StrategyPipeline._01_get_obs_dataframe(obs, step, num_agents)
print(df_s.shape, planet_disp.shape)
print(df_s.columns.tolist())

In [ ]:
pa = StrategyPipeline._02_get_all_opportunities(df_s, planet_disp, player_id=0)
print("pa shape:", pa.shape)
print("pa columns:", pa.columns.tolist())
if not pa.empty:
    print(pa[["id_src", "step_src", "id_tgt", "step_tgt", "ships_sent", "angle"]].head(10))

In [ ]:
safe = StrategyPipeline._03_filter_collision(pa)
print("safe shape:", safe.shape)
print("pa shape:", pa.shape)
assert safe.shape[0] <= pa.shape[0], "filter should only remove rows"
assert "final_angle" in safe.columns
print("_03 assertions passed")